In [ ]:
!pip -q install numpy geopandas folium branca rapidfuzz openpyxl requests shapely

In [ ]:
import io
import numpy as np
import re
import json
import requests
import pandas as pd
import geopandas as gpd

from rapidfuzz import process, fuzz

import folium
from folium import GeoJson
from folium.features import GeoJsonTooltip
from branca.colormap import LinearColormap, StepColormap

from shapely.ops import unary_union


In [ ]:
GEOJSON_PATH = "INDIAN_SUB_DISTRICTS.geojson"
india_gdf = gpd.read_file(GEOJSON_PATH)

print(india_gdf.head())

In [ ]:
state_name_col = "stname"

maharashtra_gdf = india_gdf[
    india_gdf[state_name_col].str.lower() == "maharashtra"
]

maharashtra_gdf = maharashtra_gdf.reset_index(drop=True)


print(maharashtra_gdf.columns.tolist())
print(maharashtra_gdf.head())
print(f"Number of sub-districts in Maharashtra: {len(maharashtra_gdf)}")
maharashtra_gdf.tail()

In [ ]:

mumbai_sub_gdf = maharashtra_gdf[maharashtra_gdf["sdtname"] == 'Mumbai Suburban']

print(mumbai_sub_gdf.head())
for col in mumbai_sub_gdf.columns:
    print(f"{col}: {mumbai_sub_gdf[col].dtype}")

# Dissolve all 3 rows into 1, unioning the geometries
mumbai_sub_merged = mumbai_sub_gdf.dissolve(
    by="sdtcode11",         # group by the subdist code (same for all 3)
    aggfunc={
        "Shape_Length": "sum",   # sum lengths
        "Shape_Area": "sum",     # sum areas
        # Keep first value for identifier columns (all same across rows)
        "OBJECTID": "first",
        "stcode11": "first",
        "dtcode11": "first",
        "stname": "first",
        "dtname": "first",
        "sdtname": "first",
        "Subdt_LGD": "first",
        "Dist_LGD": "first",
        "State_LGD": "first",
    }
).reset_index()

print(mumbai_sub_merged.shape)       # should be (1, ...)
print(mumbai_sub_merged["Shape_Area"].values)   # should be ~490M
print(mumbai_sub_merged.geometry)
print(mumbai_sub_merged.head())

In [ ]:
maharashtra_gdf = maharashtra_gdf[maharashtra_gdf["sdtname"] != 'Mumbai Suburban']
maharashtra_gdf = pd.concat([maharashtra_gdf, mumbai_sub_merged], ignore_index=True)

maharashtra_gdf = maharashtra_gdf.reset_index(drop=True)
maharashtra_gdf = maharashtra_gdf.sort_values(by="sdtcode11").reset_index(drop=True)
print(maharashtra_gdf.shape)
maharashtra_gdf.tail()
print("Total Area of Maharashtra sub-districts:", maharashtra_gdf["Shape_Area"].sum())

In [ ]:
sub_district_name_col = "sdtname"
census_data_path = "pop_area.csv"

census_df = pd.read_csv(census_data_path)

print(census_df.columns.tolist())
print(census_df.head())
print(f"Number of rows in census data: {len(census_df)}")

In [ ]:
census_df = census_df[census_df["Total/Rural/Urban"].str.lower() == "total"]
maharashtra_census_df = census_df[census_df["State Code"] == 27]
maharashtra_census_df = maharashtra_census_df[maharashtra_census_df["Administrative Unit"].str.lower() == "sub-district"]

maharashtra_census_df = maharashtra_census_df.reset_index(drop=True)

# print(maharashtra_census_df.tail())
# print(f"Number of rows in maharashtra census data: {len(maharashtra_census_df)}")

# print(f"Total area of maharashtra sub-districts: {maharashtra_census_df['Area (in sq. km.)'].sum()}")

# print(f"Top 10 sub-districts by population:\n{maharashtra_census_df.sort_values(by='Total Population', ascending=False).head(10)}")
maharashtra_census_df.loc[maharashtra_census_df["District Code"] == "518", ["Name", "Sub District Code"]] = "Mumbai Suburban", "05927"
maharashtra_census_df.loc[maharashtra_census_df["District Code"] == "519", ["Name", "Sub District Code"]] = "Greater Mumbai", "05926"

maharashtra_census_df = maharashtra_census_df.sort_values(by="District Code").reset_index(drop=True)
print(maharashtra_census_df.head())
print(maharashtra_census_df.tail())
print(f"Number of rows in maharashtra census data: {len(maharashtra_census_df)}")



In [ ]:
# drop unnecessary columns from maharashtra_census_df

print(f"Columns in maharashtra_census_df before dropping: {maharashtra_census_df.columns.tolist()}")
maharashtra_census_df = maharashtra_census_df.drop(columns=[
    "State Code",
    "District Code",
    "Total/Rural/Urban",
    "Administrative Unit",
    "Number of Inhabited villages",
    "Number of Uninhabited villages",
    "Number of towns",
    "Number of households",
    "Male Population",
    "Female Population"
])

print(f"Columns in maharashtra_census_df after dropping: {maharashtra_census_df.columns.tolist()}")


In [ ]:
# Drop unnecessary columns from maharashtra_gdf

print(f"Columns in maharashtra_gdf before dropping: {maharashtra_gdf.columns.tolist()}")

maharashtra_gdf = maharashtra_gdf.drop(columns=[
    "OBJECTID",
    "stcode11",
    "dtcode11",
    "stname",
    "dtname",
    "stname",
    "dtname",
    "Subdt_LGD",
    "Dist_LGD",
    "State_LGD"
])

print(f"Columns in maharashtra_gdf after dropping: {maharashtra_gdf.columns.tolist()}")

In [ ]:
print(f"Number of rows in maharashtra_gdf: {len(maharashtra_gdf)}")
print(f"Number of rows in maharashtra_census_df: {len(maharashtra_census_df)}")

# Compare side by side the maharashtra_gdf and maharashtra_census_df to see if they match

print("maharashtra_gdf:")
print(maharashtra_gdf.head())
print("\nmaharashtra_census_df:")
print(maharashtra_census_df.head())

# Merge



In [ ]:
merged_df = maharashtra_gdf.merge(maharashtra_census_df, left_on="sdtcode11", right_on="Sub District Code", how="outer", indicator=True)

print(f"Number of rows in merged_df: {len(merged_df)}")


merged_df.drop(columns=["_merge", "Name"], inplace=True)
merged_df = merged_df.sort_values(by="sdtcode11").reset_index(drop=True)
print(merged_df.head())

In [ ]:
# Ensure data is in WGS84 for folium
map_gdf = merged_df.to_crs(epsg=4326).copy()

density_col = "Population per sq. km."
map_gdf[density_col] = pd.to_numeric(map_gdf[density_col], errors="coerce")

# Use log scaling for color only
density_values = map_gdf[density_col].dropna()
log_min = np.log1p(density_values.min())
log_max = np.log1p(density_values.max())

base_colors = ["#ffffcc", "#fed976", "#fecc5c", "#fd8d3c", "#ef7014", "#e34d48", "#f03b20", "#e31a1c", "#800026"]

def density_to_color(value):
    if pd.isna(value) or log_max == log_min:
        return "#808080"
    norm = (np.log1p(value) - log_min) / (log_max - log_min)
    # Clamp norm to [0, 1]
    norm = max(0, min(1, norm))
    # Get the color from base_colors based on norm
    idx = int(norm * (len(base_colors) - 1))
    return base_colors[idx]

# Create colormap for the legend
legend_vmin = 0
legend_vmax = int(np.ceil(density_values.max() / 10000.0) * 10000)

colormap = LinearColormap(
    colors=base_colors,
    vmin=legend_vmin,
    vmax=legend_vmax,
    caption="Population density (per sq. km.)"
)

print(f"Colormap created: vmin={legend_vmin}, vmax={legend_vmax}")

# Center the map on Maharashtra and constrain the viewport to its bounds
minx, miny, maxx, maxy = map_gdf.total_bounds

padding = 0.4  # degrees of padding to add around the bounds

minx -= padding  # Adjust the minimum longitude to provide some padding on the left side
miny -= padding  # Adjust the minimum latitude to provide some padding on the bottom side
maxx += padding  # Adjust the maximum longitude to provide some padding on the right side
maxy += padding  # Adjust the maximum latitude to provide some padding on the top side
m = folium.Map(
    location=[(miny + maxy) / 2, (minx + maxx) / 2],
    zoom_start=7,
    tiles="cartodbpositron",
    max_bounds=True,
    min_lat=miny,
    max_lat=maxy,
    min_lon=minx,
    max_lon=maxx,
    min_zoom=7
)
m.fit_bounds([[miny, minx], [maxy, maxx]])

def style_function(feature):
    value = feature["properties"].get(density_col)
    return {
        "fillColor": density_to_color(value),
        "color": "black",
        "weight": 0.5,
        "fillOpacity": 0.75,
    }

geojson = folium.GeoJson(
    map_gdf,
    name="Maharashtra Sub-districts",
    style_function=style_function,
    tooltip=GeoJsonTooltip(
        fields=["sdtname", "Total Population", "Area(In sq. km)", density_col],
        aliases=["Sub-district", "Population", "Area (sq. km)", "Density"],
        localize=True,
        sticky=False
    ),
)

geojson.add_to(m)
colormap.add_to(m)
folium.LayerControl().add_to(m)

m.save("index.html")